In [63]:
yuv_video_path = '/home/filelele/personal_project/autonomous-rover/src/Phone/camera_calibrate/recording_20260914_015215.yuv'
extracted_frames_folder = '/home/filelele/personal_project/autonomous-rover/src/Phone/camera_calibrate/640x480-highquality_distortion_correction-0.4mfocus/'
width = 640
height = 480

# chessboard properties
columns = 11
rows = 8
grid_size = 1


In [4]:
import matplotlib.pyplot as plt
import cv2 as cv
import numpy as np
import glob

In [ ]:
import os
def extract_yuv_frames(yuv_path, output_dir, width, height):
    os.makedirs(output_dir, exist_ok=True)
    
    frame_size = int(width * height * 1.5)
    frame_idx = 0

    with open(yuv_path, 'rb') as f:
        while True:
            raw_bytes = f.read(frame_size)
            if len(raw_bytes) < frame_size:
                break
            
            yuv = np.frombuffer(raw_bytes, dtype=np.uint8)
            
            yuv = yuv.reshape((int(height * 1.5), width))
            
            bgr = cv.cvtColor(yuv, cv.COLOR_YUV2BGR_I420)
            
            out_name = os.path.join(output_dir, f"frame_{frame_idx:04d}.png")
            cv.imwrite(out_name, bgr)
            frame_idx += 1

    print(f"Extracted {frame_idx} frames into '{output_dir}'.")

extract_yuv_frames(yuv_video_path, extracted_frames_folder, width, height)

Extracted 1785 frames into '/home/filelele/personal_project/autonomous-rover/src/Phone/camera_calibrate/640x480-highquality_distortion_correction-0.4mfocus/'.


In [8]:
object_points_matrix = np.zeros(((columns - 1)*(rows - 1),3), np.float32)
object_points_matrix[:,:2] = np.mgrid[0:(columns-1), 0:(rows-1)].T.reshape(-1,2) * grid_size

In [9]:
object_points_matrix

array([[0., 0., 0.],
       [1., 0., 0.],
       [2., 0., 0.],
       [3., 0., 0.],
       [4., 0., 0.],
       [5., 0., 0.],
       [6., 0., 0.],
       [7., 0., 0.],
       [8., 0., 0.],
       [9., 0., 0.],
       [0., 1., 0.],
       [1., 1., 0.],
       [2., 1., 0.],
       [3., 1., 0.],
       [4., 1., 0.],
       [5., 1., 0.],
       [6., 1., 0.],
       [7., 1., 0.],
       [8., 1., 0.],
       [9., 1., 0.],
       [0., 2., 0.],
       [1., 2., 0.],
       [2., 2., 0.],
       [3., 2., 0.],
       [4., 2., 0.],
       [5., 2., 0.],
       [6., 2., 0.],
       [7., 2., 0.],
       [8., 2., 0.],
       [9., 2., 0.],
       [0., 3., 0.],
       [1., 3., 0.],
       [2., 3., 0.],
       [3., 3., 0.],
       [4., 3., 0.],
       [5., 3., 0.],
       [6., 3., 0.],
       [7., 3., 0.],
       [8., 3., 0.],
       [9., 3., 0.],
       [0., 4., 0.],
       [1., 4., 0.],
       [2., 4., 0.],
       [3., 4., 0.],
       [4., 4., 0.],
       [5., 4., 0.],
       [6., 4., 0.],
       [7., 4

In [64]:
images = sorted(glob.glob(extracted_frames_folder + '*.png'))
images = [images[i] for i in range(125,1716,10)]

In [65]:
len(images)

160

In [ ]:
test_img = cv.imread(images[-1])
print(test_img.shape)
plt.figure(figsize=(4,6))
plt.imshow(cv.cvtColor(test_img, cv.COLOR_BGR2RGB))

In [50]:
criteria = (cv.TERM_CRITERIA_EPS + cv.TERM_CRITERIA_MAX_ITER, 60, 0.0001)

In [51]:
object_points = []
image_points = []
corners_not_found_images = []
for image in images:
    img = cv.imread(image)
    if img is None:
        print(f"Failed to load image: {image}")
        break
    """
    plt.figure(figsize=(10, 8))
    plt.imshow(cv.cvtColor(img, cv.COLOR_BGR2RGB))
    plt.axis('off')
    plt.show()
    """
    print(f"Processing image: {image}")
    print(f"Image shape: {img.shape}")
    gray = cv.cvtColor(img, cv.COLOR_BGR2GRAY)
    print(f"Gray image shape: {gray.shape}")
    ret, corners = cv.findChessboardCorners(gray, ((columns - 1),(rows - 1)), None)
    print(f"Chessboard corners found: {ret}, number of corners: {corners.shape[0] if ret else 0}")
    if ret == True:
        object_points.append(object_points_matrix)
        corners2 = cv.cornerSubPix(gray, corners, (11,11), (-1,-1), criteria)
        image_points.append(corners2)
        img = cv.drawChessboardCorners(img, ((columns - 1),(rows - 1)), corners2, ret)
        
        print(img.shape, img.dtype, img.max(), img.min())
        """
        plt.figure(figsize=(6, 4))
        plt.imshow(cv.cvtColor(img, cv.COLOR_BGR2RGB))
        plt.axis('off')
        plt.show()
        """
    else :
        print(f"Chessboard corners not found in image: {image}")
        corners_not_found_images.append(image)
# cv.destroyAllWindows()

Processing image: /home/filelele/personal_project/autonomous-rover/src/Phone/camera_calibrate/640x480-highquality_distortion_correction-0.4mfocus/frame_0125.png
Image shape: (480, 640, 3)
Gray image shape: (480, 640)
Chessboard corners found: True, number of corners: 70
(480, 640, 3) uint8 255 0
Processing image: /home/filelele/personal_project/autonomous-rover/src/Phone/camera_calibrate/640x480-highquality_distortion_correction-0.4mfocus/frame_0135.png
Image shape: (480, 640, 3)
Gray image shape: (480, 640)
Chessboard corners found: True, number of corners: 70
(480, 640, 3) uint8 255 0
Processing image: /home/filelele/personal_project/autonomous-rover/src/Phone/camera_calibrate/640x480-highquality_distortion_correction-0.4mfocus/frame_0145.png
Image shape: (480, 640, 3)
Gray image shape: (480, 640)
Chessboard corners found: True, number of corners: 70
(480, 640, 3) uint8 255 0
Processing image: /home/filelele/personal_project/autonomous-rover/src/Phone/camera_calibrate/640x480-highqua

In [52]:
len(corners_not_found_images)

0

In [53]:
len(images), len(object_points), len(image_points)

(160, 160, 160)

In [54]:
object_points[0][0], image_points[0][0]


(array([0., 0., 0.], dtype=float32),
 array([109.48163,  90.18887], dtype=float32))

In [55]:
object_points[0][1], image_points[0][1]

(array([1., 0., 0.], dtype=float32),
 array([143.52924,  91.12481], dtype=float32))

In [56]:
object_points[0][2], image_points[0][2]

(array([2., 0., 0.], dtype=float32),
 array([177.03261,  92.0422 ], dtype=float32))

In [57]:
print([image_points[i].shape for i in range(len(image_points))])
print([object_points[i].shape for i in range(len(object_points))])

[(70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), (70, 2), 

In [58]:
ret, mtx, dist, rvecs, tvecs = cv.calibrateCamera(object_points, image_points, gray.shape[::-1], None, None, flags=cv.CALIB_FIX_K3)

In [59]:
mtx

array([[449.23237786,   0.        , 320.39145568],
       [  0.        , 449.10996865, 239.98704651],
       [  0.        ,   0.        ,   1.        ]])

In [60]:
ret

0.26004724553779285

In [61]:
dist

array([[ 0.08806528, -0.15521762,  0.00071539,  0.00166862,  0.        ]])